# 08 · Summary statistics

The summary statistics of the flux, built bottom-up on toy boxes whose statistics are known: the 3D fields and their
statistics (Part A), skewers through them, from the gas to the flux (Part B), the 3D statistics of the overfluxes on the
lattice of skewers (Part C), and the 1D statistics: the P1D of the overfluxes next to a measurement, and the flux
contrast term by term and order by order (Part D).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag)
from skewerjax.core import LineBundle

In [ ]:
plot_dag(('lines', 'cosmology', 'skewers', 'samplers', 'fields', 'profiles', 'deposition', 'tau', 'overflux', 'stats',)); plt.show()

## Concepts

**Overflux.** From the $\tau$ of several lines on one velocity axis, the overflux (flux contrast) of every line and of
the total flux $F = \prod_j F_j$ are

$$
\delta_j = \frac{F_j}{\langle F_j\rangle} - 1,\qquad \delta_{\rm tot} = \frac{\prod_j F_j}{\langle\prod_j F_j\rangle} - 1
$$

(review notes, Sec. Overflux; `summary_stats.DeltaFieldBuilder`), with the mean $\langle\cdot\rangle$ over a whole box in
Part C and over each skewer in Part D, where $\delta_{\rm tot}$ is also split exactly into the terms of the single lines
and of their products, order by order (D2).

**1D statistics.** Along skewers of $N$ pixels of width $dv$, P1D and $\xi_{\rm 1D}$ are an exact discrete Fourier pair
(review notes; `summary_stats.OverfluxStatistics1D`),

$$
P_{ab}(k) = \frac{dv}{N}\,\tilde\delta_a(k)\,\tilde\delta_b^*(k),\qquad
\xi_{ab}(\Delta v) = \frac{1}{N}\sum_j \delta_a(v_j)\,\delta_b(v_j + \Delta v),
$$

with $\tilde\delta$ the discrete Fourier transform, $k$ in s/km and $\Delta v$ in km/s, both averaged over the skewers.

**Figures.** A measurement is drawn as points with error bars joined by a thin line of the same colour, every
realisation (box) as a thin, faint line behind it, and an expectation as a line.

**3D statistics on a grid (Parts A and C).** For one realisation $\delta$ on $N$ cells of volume $dV$ in a box of volume $V$,

$$
\hat P(\mathbf k) = \frac{\lvert\delta_{\mathbf k}\rvert^2}{V},\qquad
\hat\xi(\mathbf r) = \frac{1}{N}\sum_{\mathbf x} \delta(\mathbf x)\, \delta(\mathbf x + \mathbf r),
$$

with $\delta_{\mathbf k}$ the discrete Fourier transform times $dV$ and $\mathbf r$ every minimum-image separation of the
periodic grid (the same normalisation as P1D, in 3D; `summary_stats.GridStatistics3D`). Both are averaged in bins: of
$\lvert k\rvert$ or $\lvert r\rvert$, of $(k_\parallel, k_\perp)$ or $(r_\parallel, r_\perp)$, and of $\lvert k\rvert$ or
$r$ in wedges of $\mu = k_\parallel/\lvert k\rvert$ or $r_\parallel/r$, with the line of sight along $y$. The expectations
are exact: $\langle\hat P\rangle$ is the input $P$ averaged over the modes of each bin, and $\langle\hat\xi\rangle$ is

$$
\xi_{\rm in}(\mathbf r) = \frac{1}{V}\sum_{\mathbf k \neq 0} P(k)\, e^{i\mathbf k\cdot\mathbf r},
$$

the discrete inverse transform of the input on the same grid (no $k = 0$ mode: it integrates to zero over the box),
averaged over the separations of each bin. Every error bar below is the error of the mean over the realisations,
$\sigma/\sqrt{n}$, of its bin alone. Part A applies these estimators to the fields in comoving units; Part C to the
overfluxes on the lattice of skewers, in velocity units.

**Overview.**

- **Part A — toy fields:** `n_real` periodic boxes of side $L$ on $N^3$ cells, with a power-law plus BAO spectrum,
  isotropic; the gas of every cell; $P$ and $\xi$ of the realisations against their expectation (comoving units).
- **Part B — skewers:** a regular lattice of skewers along $y$ through every box (`SkewerRun`); $\tau$ of
  $\mathrm{Ly}\alpha$ and Si III, periodic, using as x-axis the the reference velocity-units-frame associatted to the $\mathrm{Ly}\alpha$ transition.
- **Part C — 3D statistics of the overfluxes:** $P$ and $\xi$ of the $\mathrm{Ly}\alpha$, Si III and total overflux on the
  lattice of skewers, in velocity units (comoving units on the top axes); $\xi$ of the total
  in the $(r_\parallel, r_\perp)$ plane and in wedges of $\mu$.
- **Part D — 1D statistics:** P1D of the overfluxes; P1D and $\xi_{\rm 1D}$ of the flux
  contrast on the $\mathrm{Ly}\alpha$ axis by order and term, averaged over the boxes.

**Configuration.** The configuration of the boxes is the next cell (`ToyBoxes`: the boxes, their spectrum
and their gas), and the cell after it prints every value; the skewers (`SkewerRun`) are configured at the start of Part
B. Part A reads its statistics (and the covariance of $\xi$) from the cache of `utils.paths.CACHE_DIR` when it holds this
configuration, and computes and stores them otherwise. Part B always computes its skewers.

In [ ]:
import time
from fractions import Fraction
from skewerjax.core import phys
from skewerjax.simulations import ArepoHDF5Reader, PowerLawGRF, ToyBoxes, field_statistics, xi_chi2, xi_subset_offsets
from skewerjax.simulations.toy_boxes import LINE_IDS
from skewerjax.utils.paths import CACHE_DIR, DATA_DIR
REALISATIONS = None               # None: all of them; e.g. [0, 1] for a quick check
CACHE = CACHE_DIR / "toy_boxes"   # git-ignored ($SKEWERJAX_CACHE), keyed by the configuration: Part A's statistics, Part B's skewers
L_BOX = 240
SIIII_TO_HI = 1 / 40 # n_SiIII = r n_HI
boxes = ToyBoxes(box_cMpc=L_BOX, n_cells=220, n_real=24, base_seed=1000, bao_ratio=0.002, sigma_bao=6.0, r_bao=100, siiii_to_hi=SIIII_TO_HI)

In [ ]:
# The values of the configuration (the text refers to them by their symbols)
g = boxes.grf()
idx = np.arange(boxes.n_real) if REALISATIONS is None else np.asarray(REALISATIONS)
var_bao = 1.0 - float(PowerLawGRF(g.A_s, g.k_0, g.n_s, L=g.L, N=g.shape).variance() / g.variance())
print(f"boxes: L = {L_BOX:g} cMpc, N = {boxes.n_cells} cells per side of dx = {boxes.dx:.4g} cMpc ({boxes.dx * boxes.aHf:.1f} km/s); "
      f"{idx.size} of n_real = {boxes.n_real} realisations, i = {idx.min()}..{idx.max()}, seed base_seed + i = {boxes.base_seed} + i")
print(f"P(k): A_s = {boxes.A_s:.4g} cMpc^3, k_0 = {boxes.k_0} cMpc^-1, n_s = {boxes.n_s}; sigma_delta = {float(np.sqrt(g.variance())):.3f}")
print(f"BAO: r_BAO = {boxes.r_bao} cMpc, sigma_BAO = {boxes.sigma_bao} cMpc, bao_ratio = {boxes.bao_ratio:g}: bump height "
      f"A_BAO = {g.A_bao:.3g} = {g.A_bao / g.variance():.2e} sigma_delta^2; the BAO term holds {100 * var_bao:.1f} % of the variance")
print(f"gas: n_HI = {boxes.n_bar_HI:g} (1 + delta) cm^-3, r = n_SiIII / n_HI = {Fraction(SIIII_TO_HI).limit_denominator(1000)}, "
      f"T = {boxes.T0:g} (1 + delta)^{boxes.gamma_minus_1} K; aHf = {boxes.aHf:.1f} km/s per cMpc (z = {boxes.z:g}, "
      f"Omega_m = {boxes.Omega_m}, f = {boxes.f_growth})")

## Part A — Toy fields and their statistics

### A1 · The fields

Each box is a periodic cube of side $L$ (`box_cMpc`) on $N^3$ cells (`n_cells`) of side $dx = L/N$. Realisation
$i = 0, \dots, n_{\rm real} - 1$ is `PowerLawGRF.sample(PRNGKey(base_seed + i))` (`ToyBoxes.fields(i)`): it is
regenerated from its seed whenever it is needed, and never stored.

- **Spectrum:** the power law plus the BAO term only, so the field is isotropic (no Kaiser or Fingers-of-God term):
  $P(k) = A_s (k/k_0)^{n_s - 1} + P_{\rm BAO}(k)$. The BAO term is that of notebook 04: a Gaussian bump in $\xi$ at
  $r_{\rm BAO}$ (`r_bao`), of width $\sigma_{\rm BAO}$ (`sigma_bao`) and height $A_{\rm BAO} = $ `bao_ratio` $\times A_s$,
  whose transform rides on its envelope, $P_{\rm BAO} = A_{\rm BAO}\,E(k)\,[1 + f(k)]$ with $\lvert f\rvert \le 1$, so
  $P \ge 0$ for every strength.
- **Amplitude:** $A_s$ is set by $\sigma_\delta$ (`sigma_delta`), the standard deviation of $\delta$ on the grid. The
  densities are not clipped, so $\sigma_\delta$ must keep $1 + \delta > 0$ in every cell of every realisation
  (`ToyBoxes.fields` raises otherwise); the next cell checks it. The finer the grid, the more cells reach the tail of
  the Gaussian, and the smaller $\sigma_\delta$ must be.
- **Gas:** H I and Si III are the same gas, so they share one temperature and one velocity:

$$
n_{\rm HI} = \bar n_{\rm HI}(1 + \delta),\quad n_{\rm SiIII} = r\, n_{\rm HI},\quad
T = T_0 (1 + \delta)^{\gamma - 1},\quad \mathbf v_{\mathbf k} = i\, aHf\, \frac{\mathbf k}{k^2}\, \delta_{\mathbf k},
$$

with $r$ (`SIIII_TO_HI`), $T_0$ (`T0`), $\gamma - 1$ (`gamma_minus_1`), and the linear-theory velocities of notebook 04,
$aHf$ at redshift $z$ for $\Omega_m$ and the growth rate $f$ (`z`, `Omega_m`, `f_growth`). $\bar n_{\rm HI}$ (`n_bar_HI`)
sets the mean transmission $\langle F_{{\rm Ly}\alpha}\rangle$, printed in B1 (observed: about 0.68 at $z = 3$).

In [ ]:
res = field_statistics(boxes, CACHE, realisations=REALISATIONS)        # P and xi of every realisation (A3): the cache, or computed
m1 = res["min_one_plus_delta"]
print(f"{res['n_real']} realisations ({res['time_s']:.0f} s when computed, and {res['time_cov_s']:.0f} s for the covariance of xi): "
      f"min(1 + delta) = {m1.min():.3f} (realisation {res['realisations'][m1.argmin()]}); 1 + delta > 0 in {np.sum(m1 > 0)} of {m1.size}")

### A2 · The fields of one realisation

A slice through realisation 0 at $z = 0$, the plane of the first row of skewers of Part B ($x$ across, $y$ upwards: the
line of sight of the skewers that we will throw aftewards in part B of this notebook). The two columns differ only by the
factor $r$ of the densities. The temperature, $T = T_0(1 + \delta)^{\gamma - 1}$ (A1), is a
function of the density of the top row. The velocity colour is $v_y$, the component along
the skewers; the arrows are $(v_x, v_y)$ every `ARROW_EVERY` cells.

In [ ]:
from skewerjax.utils.plotting import plot_toy_fields
ARROW_EVERY = max(1, boxes.n_cells // 32)                                    # about 32 arrows across the slice
fields0 = boxes.fields(0)
fig, axs, info = plot_toy_fields(fields0, L_BOX, z_index=0, arrow_every=ARROW_EVERY)
add_panel_letters(axs.T.ravel(), labels=["a1", "a2", "b1", "b2"])
r_frac = Fraction(SIIII_TO_HI).limit_denominator(1000)
fig.suptitle(rf"Realisation 0, slice $z = 0$: H I and Si III are one gas, with $n_{{\rm SiIII}} = {r_frac}\,n_{{\rm HI}}$, "
             r"one velocity", fontsize=22)
plt.show()
print(f"arrows every {ARROW_EVERY} cells; sigma_delta of the realisation {fields0['delta'].std():.3f}; T from {fields0['T'].min():.0f} to {fields0['T'].max():.0f} K")

### A3 · The statistics of the realisations

Each realisation is generated and measured on its grid (`field_statistics`, `GridStatistics3D`). Every panel
shows the mean over the realisations with the error of the mean, every realisation as a
faint line, and the exact expectation on top.

**$P$ and $\xi$.** $P(\lvert k\rvert)$ has one bin per mode for the first modes, then bins of a quarter of the
fundamental mode up to the end of the BAO wiggles, $k \simeq 3/\sigma_{\rm BAO}$, then logarithmic bins; $\xi$ runs out
to $r_{\rm BAO} + 4\sigma_{\rm BAO}$ when that lies beyond $L/2$, so the whole peak shows (`ToyBoxes.bin_edges`). The BAO
peak is shown as $r^2\xi(r)$ on linear axes.

In [ ]:
from skewerjax.utils.plotting import plot_isotropic_recovery, plot_kpar_kperp_maps, plot_rpar_rperp_maps
fig, axs, rec = plot_isotropic_recovery(res, xi_view="r2", r_bao=boxes.r_bao)
add_panel_letters(axs.T.ravel())
fig.suptitle(rf"{res['n_real']} realisations recover their input: $P(k)$ with its BAO wiggles, $r^2\xi(r)$ with its BAO peak", fontsize=22)
plt.show()
SUBSETS = (4, 8, 16)                                                   # disjoint subsets of realisations (reading below)
XI_WINDOW = (2 * boxes.r_bao / 3, boxes.r_bao + 4 * boxes.sigma_bao)   # the r range of the offset [cMpc]
c_all, c_win = xi_chi2(res), xi_chi2(res, r_range=XI_WINDOW)
print(f"P(|k|): chi2/dof = {rec['P_chi2_dof']:.2f}, max |P / P_th - 1| = {rec['P_max_rel']:.3f}")
for name, c in (("all bins", c_all), (f"{XI_WINDOW[0]:.0f} < r < {XI_WINDOW[1]:.0f} cMpc", c_win)):
    print(f"xi(|r|), {name} ({c['dof']} bins): chi2 diagonal = {c['chi2_diag']:.1f}, full covariance = {c['chi2_full']:.1f} "
          f"(p = {c['p_full']:.2f}); first eigenvector {100 * c['mode1_share']:.0f} % of the variance; the offset lies "
          f"{100 * c['offset_top3_share']:.0f} % along the 3 largest, at {np.round(np.abs(c['pulls_top3']), 2)} sigma")
print(f"chi2 of each realisation (all bins, covariance of one realisation): mean {c_all['chi2_each'].mean():.1f} for "
      f"{c_all['dof']} bins, from {c_all['chi2_each'].min():.0f} to {c_all['chi2_each'].max():.0f}")
print(f"offset of r^2 xi from its expectation, averaged over {XI_WINDOW[0]:.0f} < r < {XI_WINDOW[1]:.0f} cMpc [cMpc^2]:")
for o in xi_subset_offsets(res, SUBSETS, r_range=XI_WINDOW):
    print(f"  subsets of {o['size']:2d}: " + ", ".join(f"{x:+.3f}" for x in o["offsets"]) + f"   (standard deviation {o['sigma']:.3f})")

*Reading the figure.* The measurement follows the BAO wiggles of $P$ bin by bin, and the peak of $r^2\xi$ at
$r_{\rm BAO}$; $\xi$ turns negative on large scales because the box has no $k = 0$ mode. Devaitions from
the expectation in the real 2pcf are due to cosmic variance, not a bias:

- **The error bars** are the error of the mean of each bin alone, $\sigma_a/\sqrt{n_{\rm real}}$: the diagonal of the covariance. They say nothing about how the bins move together.
- **The bins move together.** $\hat\xi(\mathbf r) = V^{-1}\sum_{\mathbf k}\hat P(\mathbf k)\cos(\mathbf k\cdot\mathbf r)$
  and, for a Gaussian field, ${\rm Cov}[\hat P(\mathbf k), \hat P(\mathbf k')] = P(k)^2\,(\delta_{\mathbf k\mathbf k'} +
  \delta_{\mathbf k,-\mathbf k'})$, so the covariance of the bins $a$ and $b$ of one realisation is exactly

$$
C_{ab} = \frac{2}{V^2}\sum_{\mathbf k} P(k)^2\, c_a(\mathbf k)\, c_b(\mathbf k),\qquad
c_a(\mathbf k) = \langle\cos(\mathbf k\cdot\mathbf r)\rangle_{\mathbf r \in a}
$$

  (`GridStatistics3D.xi_iso_covariance`). With a red spectrum, $n_s - 1 < 0$, the sum is dominated by the few lowest
  modes, whose $c_a$ change slowly from bin to bin: one fluctuation of their amplitudes shifts many bins in the same
  direction. The first eigenvector of $C$ holds most of the variance (printed). The covariance is checked on the
  realisations themselves: the $\chi^2$ of each one against $C$ averages the number of bins (printed).
- **With the full covariance** of the mean, $\chi^2 = \mathbf d^{\rm T}(C/n_{\rm real})^{-1}\mathbf d$ with
  $\mathbf d = \bar\xi - \xi_{\rm in}$, the offset is consistent with zero, over all the bins and over the window of
  the BAO peak, `XI_WINDOW` (printed, with the probability of a larger $\chi^2$). The visible offset lies almost entirely
  along the three eigenvectors of largest variance, at about one standard deviation each (printed): a coherent shift of
  many bins is one ordinary fluctuation, not many. The diagonal $\chi^2$ treats the bins as independent and misjudges
  it, too high over some ranges and too low over others.
- **With the number of realisations,** the offset behaves like one draw of that coherent fluctuation: for disjoint
  subsets of increasing size (`SUBSETS`, printed: the offset of $r^2\xi$ averaged over `XI_WINDOW`), it changes sign
  from subset to subset and stays within its standard deviation, which falls as $1/\sqrt{n}$.
- **The expectation is the generator's model:** `field_statistics` evaluates `GridStatistics3D.theory` on
  `boxes.grf().power_grid()`, the very array that scales the white noise in `PowerLawGRF.sample` (BAO term included), on
  the same discrete modes, so it is exact bin by bin. $P$, whose modes are independent, shows no offset.

**The $(k_\parallel, k_\perp)$ and $(r_\parallel, r_\perp)$ planes.** The same statistics resolved along and across the
line of sight ($y$). $P$ on a logarithmic colour scale and both axes logarithmic; $r^2\xi$ on a linear colour scale,
linear axes up to half the box. Iso-contours: of $P$ at every decade, of $r^2\xi$ at evenly spaced levels; the
expectation's dashed, the measured ones solid. The rays from the origin are the boundaries of the wedges of $\mu$ of the
next figure (`MU_WEDGES`).

In [ ]:
fig, axs, chi2_P = plot_kpar_kperp_maps(res)
add_panel_letters(axs)
fig.suptitle(r"$P(k_\parallel, k_\perp)$: circles in the plane, measured on the expectation (an isotropic field)", fontsize=22)
plt.show()
print(f"chi2/dof per bin: P map {chi2_P:.2f}")

In [ ]:
fig, axs, chi2_xi = plot_rpar_rperp_maps(res, r_bao=boxes.r_bao)
add_panel_letters(axs)
fig.suptitle(rf"$r^2\xi(r_\parallel, r_\perp)$: the same in real space, the BAO ring at $r_{{\rm BAO}} = {boxes.r_bao:g}$ cMpc",
             fontsize=22)
plt.show()
print(f"chi2/dof per bin: xi map {chi2_xi:.2f}")
# The ring and what lies beneath it, wedge by wedge at r_BAO: the bump = xi - xi_nw (xi_nw: the expectation without the
# BAO wiggle) and the background xi_nw
a = res["axes"]
j = np.argmin(np.abs(a["r_mu"] - boxes.r_bao), axis=1)
w = np.arange(len(j))
th, nw = res["theory"]["xi_mu"][w, j], res["theory_nw"]["xi_mu"][w, j]
meas, err = res["xi_mu"][w, j] - nw, res["xi_mu_err"][w, j]
print(f"at r = r_BAO, in units of A_BAO = {g.A_bao:.3g}:")
print("  wedge of mu      bump expected   bump measured     background xi_nw")
for k in w:
    print(f"  {a['mu_edges'][k]:.2f} - {a['mu_edges'][k + 1]:.2f}       {(th[k] - nw[k]) / g.A_bao:6.3f}      "
          f"{meas[k] / g.A_bao:6.2f} +- {err[k] / g.A_bao:4.2f}      {nw[k] / g.A_bao:+6.2f}")
k_nyq = np.pi / boxes.dx
print(f"P_BAO at the Nyquist frequency pi/dx = {k_nyq:.2f} cMpc^-1, relative to its value at the fundamental mode: "
      f"{float(g.bao_power(k_nyq) / g.bao_power(2 * np.pi / L_BOX)):.1e}")

**Wedges.** $\mathrm{Ly}\alpha$ 3D analyses show the anisotropy in wedges of $\mu$. In here we show $r^2\xi(r)$ against $r$
on linear axes, data as points with error bars in one colour per wedge and the model as solid lines (here the
expectation of each wedge). In infinite space an isotropic field puts every wedge on the same curve.

In [ ]:
from skewerjax.utils.plotting import plot_xi_mu_wedges
fig, axs, chi2_rmu = plot_xi_mu_wedges(res, r_bao=boxes.r_bao)
add_panel_letters(axs)
fig.suptitle(r"$r^2\xi(r, \mu)$ as in DESI DR2: the same BAO bump in every wedge, on a background set by the box", fontsize=22)
plt.show()
print("chi2/dof per wedge (diagonal):", np.round(chi2_rmu, 2))

## Part B — Skewers

### B1 · Skewers through each box

**Layout.** In every box, skewers run along $y$ through the centres of the cell columns (the nodes $x = i_x\,dx$,
$z = i_z\,dx$ of `grf_cells`), from $y = 0$ to $y = L$, so their first and last points are the same place of the periodic
box. `SkewerBundle`, the cells of `ToyBoxes.cells` and `compute_optical_depth` give $\tau$ of $\mathrm{Ly}\alpha$ and
Si III 1206, periodic over one box (`velocity_period=True`) and both on the $\mathrm{Ly}\alpha$ axis
(`line_frame="reference"`).

**The set-up, `SkewerRun`** (next cell, which prints what follows from it), with $p$ = `points_per_cell` and
$N_{\rm pix}$ = `n_pix`:

| Argument | What it controls | Result or cost |
|---|---|---|
| `stride` $= (s_x, s_z)$ | the cell columns that become skewers: every $s_x$-th in $x$ and every $s_z$-th in $z$, from column 0, exactly through the cell centres $x = i\,s_x\,dx$, $z = j\,s_z\,dx$; $\lceil N/s_x\rceil \lceil N/s_z\rceil$ skewers per box, $s\,dx$ apart | the result: the number of skewers (the noise of every statistic) and the transverse sampling of Part C (Nyquist frequency $\pi/(s\,dx)$); the cost is linear in the number of skewers |
| `points_per_cell` | $p$, the points of the path per cell length along $y$, $N p + 1$ per skewer: every cell is cut into pieces of width $W = dx/p$ | the result until it converges ($\tau$ sums the pieces at their centroids, accurate while $W \ll b$: the resolution check below); the cost of $\tau$ is linear in $p$ |
| `n_pix` | $N_{\rm pix}$ velocity pixels over one period, of width $dv = v_{\rm per}/N_{\rm pix}$: the velocity resolution | the result: the $k$ range of P1D, up to $\pi/dv$, and the line-of-sight resolution of Part C; the cost of $\tau$ is linear in $N_{\rm pix}$ |
| `batch` | the skewers per call of `cell_pieces` and `compute_optical_depth` | the memory only, never the result |

Here $v_{\rm per}$ is the velocity span of one box along a skewer, and $u = v_{\rm per}/L$ converts comoving distances to
velocities (Part C).

In [ ]:
from skewerjax.simulations import SkewerRun, load_skewer_run, run_skewers
SETUP = SkewerRun(stride=(4, 4), points_per_cell=32, n_pix=200)     # the skewers (table above)
der = SETUP.derived(boxes)
n_x, n_z = der["n_x"], der["n_z"]
cols = boxes.columns(SETUP.stride)
print(f"skewers: stride {tuple(SETUP.stride)} -> {n_x} x {n_z} = {der['per_box']} per box, {der['per_box'] * idx.size} in all "
      f"({idx.size} boxes)")
print(f"points per skewer: {der['points_per_skewer']} ({SETUP.points_per_cell} per cell); pieces of W = {boxes.dx / SETUP.points_per_cell:.4f} cMpc "
      f"= {der['piece_kms']:.3f} km/s")
print(f"pixels: {SETUP.n_pix} over the period v_per = {der['period_kms']:.0f} km/s: dv = {der['dv_kms']:.2f} km/s, "
      f"P1D up to pi / dv = {np.pi / der['dv_kms']:.4f} s/km")
print(f"u = v_per / L = {der['kms_per_cMpc']:.2f} km/s per cMpc (aH = {der['aH_start']:.2f} at y = 0, {der['aH_end']:.2f} at y = L); "
      f"transverse spacing s dx = {der['d_perp_cMpc'][0]:.3g} x {der['d_perp_cMpc'][1]:.3g} cMpc = "
      f"{der['d_perp_kms'][0]:.0f} x {der['d_perp_kms'][1]:.0f} km/s")
print(f"lattice of skewers {'periodic: Part C runs' if SETUP.periodic(boxes) else 'not periodic: no Part C'}; transverse "
      f"Nyquist frequency pi / (s dx u) = {np.pi / der['d_perp_kms'].max():.2e} s/km")

**Resolution.** A cell spans $dx\,aH$ in velocity, much more than the thermal width $b = \sqrt{2k_BT/m}$ of Si III,
the narrowest line. The check of `compute_optical_depth` asks that only a small fraction of the column (its tolerance)
lie in pieces wider than $b/4$, so `points_per_cell` must keep the pieces narrow enough in the coldest cells.

**Streaming.** Each box is generated, its skewers computed, its statistics measured (the 3D statistics of Part C and the
1D statistics of Part D) and only those stored, one file per box (`run_skewers`), together with the skewers of box 0 for
the figures. The notebook always computes the skewers of its configuration (`overwrite=True`: a folder of the same
configuration is emptied first), so it never depends on an earlier run. `python scripts/runs/toy_boxes.py` runs the same
step, resumable and shared between processes, for long runs.

In [ ]:
from pathlib import Path
t0 = time.perf_counter()
folder = run_skewers(boxes, CACHE, SETUP, realisations=REALISATIONS, overwrite=True)   # always recomputed
run = load_skewer_run(folder, realisations=REALISATIONS)
t_skewers = time.perf_counter() - t0
n_box, n_sk = run["flux"]["n_real"], len(cols)
size = sum(p.stat().st_size for p in Path(folder).glob("*.npz")) / 1e6
dev = {d: run["time_s"][run["device"] == d].sum(1) for d in np.unique(run["device"])}
period = float(run["skewers"][0]["period"])
lines = LineBundle(list(LINE_IDS), ref_line_id=LINE_IDS[0])
b_T0 = {lid: np.sqrt(2 * phys.kB * boxes.T0 / m) / 1e3 for lid, m in zip(LINE_IDS, np.asarray(lines.mass_amu))}
print(f"{n_box} boxes x {n_sk} skewers ({n_x} x {n_z}) in {t_skewers:.0f} s; cache {size:.0f} MB in "
      f"{Path(folder).relative_to(CACHE_DIR.parent) if Path(folder).is_relative_to(CACHE_DIR.parent) else folder}")
print("time per box: " + ", ".join(f"{d} {t.mean():.0f} s ({t.size} boxes)" for d, t in dev.items())
      + f"; per box: fields {run['time_s'][:, 0].mean():.1f} s, pieces {run['time_s'][:, 1].mean():.1f} s, "
      f"tau {run['time_s'][:, 2].mean():.0f} s, statistics {run['time_s'][:, 3].mean():.1f} s")
print(f"axis: period {period:.0f} km/s, {SETUP.n_pix} pixels of dv = {period / SETUP.n_pix:.2f} km/s; a cell spans "
      f"{period / boxes.n_cells:.1f} km/s ({SETUP.n_pix / boxes.n_cells:g} pixels), a piece {period / boxes.n_cells / SETUP.points_per_cell:.2f} "
      f"km/s; b at T0: " + ", ".join(f"{lid} {b:.1f} km/s" for lid, b in b_T0.items()))
res_check = run["resolution"]
print(f"resolution check: max W / b = {np.round(res_check[..., 0].max(axis=(0, 1)), 3)}, column fraction above b/4 at most "
      f"{np.round(res_check[..., 1].max(axis=(0, 1)), 6)} (Lya, Si III); warnings: {sum(len(w) for w in run['warnings'].values())}")
print(f"<F>: Lya {run['mean_F'][:, 0].mean():.4f}, Si III {run['mean_F'][:, 1].mean():.4f}, total {run['mean_F_total'].mean():.4f} "
      f"(scatter between boxes {run['mean_F'][:, 0].std():.1e})")

### B2 · One skewer, from the gas to the flux contrast

The skewer of box 0 nearest to $x = L/2$ at $z = 0$: the vertical line through the middle of the slices of A2. Rows (a)
and (b) are along the comoving position $y$ (the $N$ cell values, as steps); rows (c) and (d) along the velocity of the
$\mathrm{Ly}\alpha$ axis, on which Si III is evaluated at $v + \Delta v$ and wrapped into the period. The overdensities
(b) of H I and Si III are identical by construction (one gas, with one temperature); their absorption is not: Si III is
narrower ($b \propto m^{-1/2}$) and displaced by the separation $\Delta v$ of the two lines, wrapped into the period
(printed below). The total flux is the product of the two, and every contrast is taken against the mean along this
skewer.

In [ ]:
from skewerjax.utils.plotting import plot_toy_skewer
sk = run["skewers"][0]
on_z0 = np.flatnonzero(sk["columns"][:, 1] == 0)
j = int(on_z0[np.argmin(np.abs(sk["columns"][on_z0, 0] - boxes.n_cells // 2))])   # the column nearest to x = L/2 at z = 0
ix, iz = sk["columns"][j]
n_col = {sp: fields0["n"][sp][ix, :, iz] for sp in ("HI", "SiIII")}
fig, axs = plot_toy_skewer(L_BOX, n_col, sk["v"], sk["tau"][:, j])
add_panel_letters(axs)
fig.suptitle(rf"One skewer ($x = {ix * boxes.dx:.1f}$, $z = {iz * boxes.dx:.1f}$ cMpc): one gas, two lines of different "
             r"width and position on the Ly$\alpha$ axis", fontsize=22)
plt.show()
dv_12 = float(lines.delta_v[1])
print(f"Si III - Lya separation Delta v = {dv_12:.0f} km/s, wrapped into the period of {period:.0f} km/s: "
      f"{(dv_12 + period / 2) % period - period / 2:.0f} km/s; b(Lya) / b(Si III) = {b_T0[LINE_IDS[0]] / b_T0[LINE_IDS[1]]:.2f}")

## Part C — 3D statistics of the overfluxes

The overflux of each line, and of their product, on the lattice of skewers of Part B is a 3D field
periodic in the three directions. Its statistics use the estimators of Part A, here **in velocity units**: $k$ in s/km, $r$ in
km/s, with the comoving units on the top axes.

- **The field.** Per box, $\delta_X = F_X/\langle F_X\rangle_{\rm box} - 1$ for $X$ = $\mathrm{Ly}\alpha$, Si III and the
  total, $F = F_{{\rm Ly}\alpha}F_{\rm SiIII}$, with the mean over the whole box rather than over each skewer (a
  per-skewer mean would remove the $k_\parallel = 0$ mode of every skewer and bias $\xi$ across the line of sight).
- **Velocity units across the line of sight.** Along a skewer the line centre of a gas element at rest is at
  $v = c\ln(a_\dagger/a)$, so $dv/dy = aH$, which changes along the skewer with $a(y)$. One box spans
  $v_{\rm per} = \int_0^L aH\,dy$, and the transverse comoving separations are converted with the same Hubble factor,
  averaged over the skewer:

$$
\Delta v_\perp = u\,\Delta x_\perp,\qquad u = \frac{v_{\rm per}}{L} = \frac{1}{L}\int_0^L a H\,dy,
$$

  so the lattice is a cube of side $v_{\rm per}$ and the skewers are $u\,s\,dx$ apart (B1 prints $u$, $aH$ at both ends
  of the skewer, and the spacing). The top axes convert back with the same factor, $x = v/u$ and $k_x = u\,k_v$
  (`utils.plotting.comoving_axis`).
- **What the lattice resolves.** Along the line of sight the pixels resolve $k_\parallel$ up to $\pi/dv$; across it the
  spacing resolves $k_\perp$ up to the transverse Nyquist frequency $\pi/(u\,s\,dx)$. Every mode of the lattice enters
  the $\lvert k\rvert$ bins, up to the corner of its mode grid, and every separation of two pixels the $\lvert r\rvert$
  bins, up to the corner of the lattice, $\sqrt{3}\,v_{\rm per}/2$; C1 shades where the bins lack directions.
- **Errors.** Points with error bars are the mean over the boxes and its error, as in Part A, with every box as a faint
  line. The flux has no exact expectation to draw.

### C1 · Power spectrum and correlation function of the overfluxes

The counterpart of the $P$ and $r^2\xi$ figure of A3, with its estimators (every mode in $\lvert k\rvert$ bins, every pair
of pixels in $\lvert r\rvert$ bins) and styles.

In [ ]:
from skewerjax.utils.plotting import plot_overflux_3d
u = period / L_BOX                                                       # km/s per cMpc (B1): v = u x, k_v = k_x / u
d_perp = der["d_perp_kms"].max()                                         # the transverse spacing of the lattice [km/s]
NAMES = {"HI-LyA": r"Ly$\alpha$", "SiIII-1206": "Si III 1206", "total": r"total, Ly$\alpha$ $\times$ Si III"}
fig, axs = plot_overflux_3d(run["flux3d"], run["axes3d"], kms_per_cMpc=u, r_bao=boxes.r_bao * u, k_mu_full=np.pi / d_perp,
                            r_mu_full=(d_perp, period / 2), names=NAMES)
add_panel_letters([axs["P"], axs["xi"]])
fig.suptitle(rf"The overfluxes in 3D, {run['flux3d']['total']['n_real']} boxes: one gas, so the total exceeds its two lines on large scales", fontsize=22)
plt.show()
ax3, f3 = run["axes3d"], run["flux3d"]
PT = f3["total"]["P_iso"]
print(f"lattice {n_x} x {SETUP.n_pix} x {n_z} per box, side v_per = {period:.0f} km/s: |k| from {ax3['k_edges'][0]:.2e} to "
      f"{ax3['k_edges'][-1]:.2e} s/km in {ax3['k_edges'].size - 1} bins, every direction up to pi / (u s dx) = {np.pi / d_perp:.2e} s/km; "
      f"|r| up to {ax3['r_edges'][-1]:.0f} km/s, every direction from {d_perp:.0f} to {period / 2:.0f} km/s")
print(f"r_BAO u = {boxes.r_bao * u:.0f} km/s, sigma_BAO u = {boxes.sigma_bao * u:.0f} km/s; BAO bump height A_BAO / sigma_delta^2 = "
      f"{g.A_bao / g.variance():.1e}")
print("P_line / P_tot at the lowest |k|: " + ", ".join(f"{f} {f3[f]['P_iso'][0] / PT[0]:.2f}" for f in LINE_IDS))

*Reading the figure.*

- **The range.** The $\lvert k\rvert$ bins used to stop at the transverse Nyquist frequency $\pi/(u\,s\,dx)$, only
  $N/(2s)$ fundamental modes for a stride of $s$ cells: a handful of bins. They now hold every mode of the lattice, out to
  $\pi/dv$ along the line of sight, and the $\lvert r\rvert$ bins every separation, out to the corner of the lattice
  (printed). Beyond $\pi/(u\,s\,dx)$ (shaded) the transverse modes stop, so a $\lvert k\rvert$ bin lacks the directions
  closest to the transverse plane, more of them as $\lvert k\rvert$ grows, and $P(\lvert k\rvert)$ tends to the power along
  the line of sight. In $r$ the same happens below the transverse spacing (line-of-sight separations only) and beyond
  $v_{\rm per}/2$ (the separations towards the corners of the lattice only, shaded).
- **$P$.** At low $k$ the total holds more power than its two lines together (their ratios add up to less than one,
  printed): H I and Si III are one gas, so their overfluxes are correlated, and for $k_\parallel \ll 2\pi/\Delta v$ the
  cross term, $\propto\cos(k_\parallel\Delta v)$, adds in phase. Where a bin holds every direction the cross term averages
  out over them; beyond the transverse Nyquist frequency the directions left lie close to the line of sight, and its
  oscillation in $k_\parallel$ shows in the total. Si III's share is set by its abundance $r$ (A1) and its mean
  transmission (B1).
- **$\xi$.** $r^2\xi$ of every overflux is positive on small scales and turns negative towards half the period, like the
  density's in Part A (no $k = 0$ mode); the total's exceeds the sum of the lines' by their cross-correlation. The points
  jump from bin to bin: the lattice is coarse across the line of sight, so the separations of an $\lvert r\rvert$ bin take
  a few values of $\mu$, which change from bin to bin, and the $\xi$ of the flux depends strongly on $\mu$ (C2, C3). The
  BAO bump does not show: its height $A_{\rm BAO}$ is a small fraction of $\sigma_\delta^2$ (printed), below the noise of
  $n_x n_z$ skewers per box. Beyond $v_{\rm per}/2$ the corner separations, with a large $r^2$, dwarf the rest: the axis
  is then symmetric-logarithmic, linear between the dashed lines.

### C2 · The $(r_\parallel, r_\perp)$ plane of the total overflux

$r^2\xi(r_\parallel, r_\perp)$ of the total overflux, as the measured panel of the $(r_\parallel, r_\perp)$ maps of A3: a
linear, diverging colour scale centred on 0, iso-$r^2\xi$ contours, and the boundaries of the wedges of $\mu$ of C3 as
rays from the origin. $r_\parallel$ is binned in pixels along the skewers and $r_\perp$ in the transverse spacing of the
lattice, out to half the period; the cMpc of the top and right axes use the factor $u$ of the line of sight.

In [ ]:
from skewerjax.utils.plotting import plot_overflux_rpar_rperp
tot = dict(run["flux3d"]["total"], axes=run["axes3d"])                    # the total overflux and its bins
fig, ax = plot_overflux_rpar_rperp(tot, kms_per_cMpc=u, r_bao=boxes.r_bao * u)
fig.suptitle(r"$r^2\xi(r_\parallel, r_\perp)$ of the total overflux: squashed along the line of sight", fontsize=22)
plt.show()
print(f"bins: r_par {np.diff(ax3['rpar_edges'])[0]:.0f} km/s, r_perp {np.diff(ax3['rperp_edges'])[0]:.0f} km/s (the transverse "
      f"spacing), out to {ax3['rpar_edges'][-1]:.0f} km/s")

*Reading the map.* Unlike the circles of A3, whose field has no velocities, the correlation of the flux is squashed
along the line of sight: the peculiar velocities of linear theory (growth rate $f$, A1; Kaiser 1987) make $r^2\xi$
positive across the line of sight and negative along it on large scales, so its contours run far from circles. The
horizontal bands are the transverse bins, one per spacing of the lattice. No BAO ring shows: its height is far below the
noise (C1). The $\mathrm{Ly}\alpha \times$ Si III peak at $r_\parallel = \Delta v$ is too faint for this scale; C3 shows it.

### C3 · Wedges of the total overflux

The wedges of A3 for the total overflux, in the same layout, in velocity units with cMpc on the top axis. Without an
expectation, the solid line is the monopole $\xi_0$ of the same $\lvert r\rvert$ bins, the isotropic part of $\xi$ from
the Legendre fit $\xi(r, \mu) = \sum_{\ell = 0, 2, 4}\xi_\ell(r)\,L_\ell(\mu)$ over the separations of each bin
(`GridStatistics3D(multipoles=True)`), and the lower panel compares each wedge with it, in units of the wedge's error:
the anisotropy of $\xi$ and its significance. Below about the transverse spacing the lattice holds line-of-sight
separations only, and $\xi_0$ is not defined (printed).

In [ ]:
fig, axs, chi2_flux_mu = plot_xi_mu_wedges(tot, r_bao=boxes.r_bao * u, unit="km/s", kms_per_cMpc=u)
add_panel_letters(axs)
fig.suptitle(r"$r^2\xi(r, \mu)$ of the total overflux: redshift space splits the wedges", fontsize=22)
plt.show()
r_ell = 0.5 * (ax3["r_mu_edges"][1:] + ax3["r_mu_edges"][:-1])
print(f"the monopole is defined from r = {r_ell[np.isfinite(f3['total']['xi_ell'][0])][0]:.0f} km/s (transverse spacing "
      f"{d_perp:.0f} km/s)")
print("chi2/dof of (wedge - monopole) per wedge (diagonal):", np.round(chi2_flux_mu, 2))

## Part D — 1D statistics

P1D and $\xi_{\rm 1D}$ along the skewers. D1 sets the P1D of the overfluxes of Part C next to a DESI measurement; D2 and
D3 decompose the flux contrast on the $\mathrm{Ly}\alpha$ axis into its orders and terms, with lines 1 =
$\mathrm{Ly}\alpha$ (H I) and 2 = Si III 1206. Every curve is the mean over the boxes and its error the error of the mean
over the boxes (the skewers of one box share its modes): points with error bars joined by thin lines, with every box as
a faint line behind them. P1D is binned with one mode per bin for the first modes and logarithmic bins after,
$\xi_{\rm 1D}$ in bins of a few pixels.

### D1 · P1D next to a DESI measurement

$k_\parallel P_{\rm 1D}(k_\parallel)/\pi$ against $k_\parallel$ [s/km], linear in $k_\parallel$ and logarithmic in $P$:
the convention and the $k_\parallel$ range of Fig. 1 of Chaves-Montero et al. (DESI DR1 $\mathrm{Ly}\alpha$ P1D,
`.project_material/Chaves-Montero`, `data_journal.tex`). The toy model's P1D is the mean over the skewers of
$(dv/N_{\rm pix})\lvert\tilde\delta(k_\parallel)\rvert^2$ of the overfluxes of Part C (the mean over the box), binned to the spacing of the
measurement (mean and error over the boxes), for the total overflux and for each line; the grey squares are the
measurement at the lowest redshift of that figure (`utils.reference_data.P1D_DESI_DR1_Z22`). This is a fiducial visual
reference only: the toy model is not calibrated to data, and it is at a single redshift $z$, not that of the measurement.

In [ ]:
from skewerjax.utils.plotting import plot_p1d_reference
from skewerjax.utils.reference_data import P1D_DESI_DR1_Z22 as P1D_REF
fig, ax = plot_p1d_reference(run["p1d"], P1D_REF, names=NAMES, k_max=P1D_REF["k_max_figure"])
fig.suptitle(rf"Toy P1D ($z = {boxes.z:g}$) next to DESI DR1 ($z = {P1D_REF['z']:g}$): same convention, uncalibrated amplitude",
             fontsize=22)
plt.show()
kc = run["p1d"]["k"]
toy = np.interp(P1D_REF["k"], kc, kc * run["p1d"]["p1d"][0] / np.pi)
print(f"reference / toy (Lya) over the reference's k range: median {np.median(P1D_REF['kP_pi'] / toy):.0f}; "
      f"Si III - Lya separation Delta v = {abs(dv_12):.0f} km/s: period 2 pi / Delta v = {2 * np.pi / abs(dv_12):.4f} s/km; "
      f"Nyquist frequency of the cells along the skewers pi / (u dx) = {np.pi / (u * boxes.dx):.4f} s/km")

*Reading the figure.* The toy model has the shape of the measurement, $k_\parallel P_{\rm 1D}/\pi$ rising at low
$k_\parallel$ and then flattening, but not its amplitude: its fluctuations are those of a Gaussian field of small
$\sigma_\delta$ on cells of side $dx$, without non-linear structure, so its P1D lies far below the data (printed). The
total oscillates in $k_\parallel$ with period $2\pi/\Delta v$ (printed): the cross term of a shifted copy,
$2\cos(k_\parallel\Delta v)\sqrt{P_{11}P_{22}}$, swings it between $(\sqrt{P_{11}} \pm \sqrt{P_{22}})^2$, a deep
modulation even for a much weaker Si III, as $\sqrt{P_{22}}$ enters. The $\mathrm{Ly}\alpha$ P1D falls steeply at the
Nyquist frequency of the grid cells along the skewers, $\pi/(u\,dx)$ (printed): the toy field has no fluctuations
shorter than a cell, and what is left beyond comes from the steps between the cells, smoothed by the thermal widths. The measurement shows oscillations of the same origin, metal lines overlapping the
$\mathrm{Ly}\alpha$ forest (the paper's Sec. 2.1), much weaker: the IGM's Si III is far less abundant than the toy's.

### D2 · The decomposition by order

**The terms.** `DeltaFieldBuilder` also forms, per skewer, the products $\delta_A = \prod_{j \in A}\delta_j$ of every
subset $A$ of lines, and the total overflux decomposes **exactly** into them (review notes, Sec. Overflux).
`OverfluxStatistics1D` gives P1D and $\xi_{\rm 1D}$ of the total, of every pair of terms $(A, B)$ and of every order
$\lvert A\rvert + \lvert B\rvert$: order $n$ collects the terms with $\lvert A\rvert + \lvert B\rvert = n$, order 0 is the
normalisation term $\mathcal{C}$, and the orders add up to the total to machine precision.

**The terms of two lines.** With two lines,

$$
F = \bar F_1 \bar F_2\, (1 + \delta_1)(1 + \delta_2),\qquad
\delta = \frac{\delta_1 + \delta_2 + \delta_1\delta_2 - C_{12}}{1 + C_{12}},\qquad C_{12} = \langle \delta_1 \delta_2 \rangle,
$$

where $C_{12}$ is the normalisation term ($\mathcal{C}$ below). Every term of $\xi_{\rm 1D}$ (real space) or P1D (Fourier
space) is a pair of subsets $A, B$ of $\{1, 2\}$, of order $\lvert A\rvert + \lvert B\rvert$:

| $A \backslash B$ | $\mathcal{C}$ (order 0) | 1 (order 1) | 2 (order 1) | 12 (order 2) |
|---|---|---|---|---|
| $\mathcal{C}$ | $+\mathcal{C}\mathcal{C}$ | 0 | 0 | $-C_{12}\mathcal{C}$ |
| 1 | 0 | $\xi_{11}$, $P_{11}$ | $\xi_{21}$, $P_{21}$ | $\xi_{(12)1}$, $P_{(12)1}$ |
| 2 | 0 | $\xi_{12}$, $P_{12}$ | $\xi_{22}$, $P_{22}$ | $\xi_{(12)2}$, $P_{(12)2}$ |
| 12 | $-\mathcal{C} C_{12}$ | $\xi_{1(12)}$, $P_{1(12)}$ | $\xi_{2(12)}$, $P_{2(12)}$ | $\xi_{(12)(12)}$, $P_{(12)(12)}$ |

After the ensemble average,

$$
\xi = \frac{-C_{12}\mathcal{C} + \xi_{11} + \xi_{22} + \xi_{12,{\rm sym}} + \xi_{1(12),{\rm sym}} + \xi_{2(12),{\rm sym}} + \xi_{(12)(12)}}{(1 + C_{12})^2},
$$

$$
P(k) = \frac{P_{11} + P_{22} + P_{12,{\rm sym}} + P_{1(12),{\rm sym}} + P_{2(12),{\rm sym}} + P_{(12)(12)} - 2\pi C_{12}^2\, \delta_D(k)}{(1 + C_{12})^2},
$$

where "sym" adds the pair and its transpose: 1/2 with 2/1, 1/12 with 12/1, and 2/12 with 12/2. In Fourier space the
constant pieces only touch the zero mode.

For a second line that is a shifted copy of the first, $\delta_2(v) \propto \delta_1(v - \Delta v)$, the cross term is
$P_{12,{\rm sym}} \propto 2\cos(k\Delta v)\, P_{11}$. It oscillates with period $2\pi/\Delta v$ in $k$ (D1, and
the cross terms of D3): the Fourier image of the $\xi_{\rm 1D}$ peak at $\Delta v$.

**By order.** Order 2 holds the Gaussian-like products and dominates; orders 3 and 4 are the non-linear corrections of
the product of the two fields. The line separation $\Delta v$, folded into the period (dashed; B2 prints both), is where
the cross terms put a peak in $\xi_{\rm 1D}$; in P1D they oscillate as $\cos(k\Delta v)$, with period $2\pi/\Delta v$
in $k$ (printed below in units of the mode spacing).

In [ ]:
from skewerjax.utils.plotting import plot_orders_compared, plot_terms_compared
ens = {r"flux contrast $\delta_F$": run["flux"]}
dv_lines = [0.0, dv_12]
fig, axs = plot_orders_compared(ens, "P", delta_v_list=dv_lines, figsize=(16, 10))
add_panel_letters(axs.T.ravel())
fig.suptitle("P1D by order: order 2 dominates; the thermal cutoff and the Si III oscillation", fontsize=22)
plt.show()
fig, axs = plot_orders_compared(ens, "xi", delta_v_list=dv_lines, figsize=(16, 10))
add_panel_letters(axs.T.ravel())
fig.suptitle(r"$\xi_{\rm 1D}$ by order: the Ly$\alpha$ $\times$ Si III peak at the folded separation (dashed)", fontsize=22)
plt.show()
e = run["flux"]
print(f"max |sum of the orders - total| / max |total|: P {np.abs(e['P_order'].sum(0) - e['P_total']).max() / np.abs(e['P_total']).max():.1e}, "
      f"xi {np.abs(e['xi_order'].sum(0) - e['xi_total']).max() / np.abs(e['xi_total']).max():.1e}")
print(f"cos(k Delta v): period 2 pi / Delta v = {2 * np.pi / abs(dv_12):.2e} s/km = "
      f"{period / abs(dv_12):.2f} mode spacings (2 pi / period)")

### D3 · Each order, term by term

The same decomposition one level down, one row per order. Every term $(A, B)$ of an order is drawn with its transpose
$(B, A)$ as A–B (their sum, which is real), on the order's total (the thick line). Order 2 holds the products 1–1, 2–2
and 1–2 (the cross term) and the normalisation cross term $\mathcal{C}$–12; orders 3 and 4 are 1–12, 2–12 and 12–12. The
order-0 term $\mathcal{C}$–$\mathcal{C}$ is not shown: it is a constant in $\xi_{\rm 1D}$, and lives at $k = 0$ only in
P1D.

In [ ]:
fig, axs = plot_terms_compared(ens, "P")
add_panel_letters(axs.ravel())
fig.suptitle("P1D of the flux contrast by order and term", fontsize=22)
plt.show()
fig, axs = plot_terms_compared(ens, "xi")
for ax in axs.ravel():
    ax.axvline(abs((dv_12 + period / 2) % period - period / 2), color=DIM, ls=":", lw=1.2)   # the separation, folded
add_panel_letters(axs.ravel())
fig.suptitle(r"$\xi_{\rm 1D}$ by order and term: the cross terms 1–2 carry the line separation (dotted)", fontsize=22)
plt.show()

*Reading the figures.* On large scales every term follows the density field, scaled (at linear order
$\delta_F \approx -\bar\tau\,\delta$, a bias); on small scales the thermal widths cut P1D, $\mathrm{Ly}\alpha$ first and
Si III later (its $b$ is smaller: B1), where the power is its 2–2 term. The peculiar velocities move the absorption, so
the flux is in redshift space. The cross terms 1–2 carry the line separation, a property of the axis, not of the gas.

The kernels of `skewerjax.summary_stats` (`p1d_kernel`, `GridStatistics3D`, `binned_mean`) are jitted; the checks
are in `tests/summary_stats/test_statistics.py` and `tests/simulations/test_toy_boxes.py`.